# 3. Features
Adds market information (implied volatility indices VIX, OVX and GVZ, the dollar index and US interest rates) and tests whether it improves HAR.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

rv = pd.read_csv("rv.csv", index_col=0, parse_dates=True)

market = yf.download(["^VIX", "^OVX", "^GVZ", "DX-Y.NYB"], start="2015-01-01")["Close"]
market = market.rename(columns={"^VIX": "vix", "^OVX": "ovx", "^GVZ": "gvz", "DX-Y.NYB": "dollar"})
market.tail()

[*********************100%***********************]  4 of 4 completed


Ticker,dollar,gvz,ovx,vix
Date,,,,
2026-10-16,101.839996,22.580000,50.740002,19.990000
2026-10-19,101.980003,22.230000,50.220001,18.440001
2026-10-20,101.730003,21.830000,52.110001,17.309999
2026-10-21,101.779999,21.360001,53.279999,16.680000
2026-10-22,101.582001,23.430000,55.619999,19.059999


Fear indices from Yahoo (VIX: S&P 500, OVX: oil, GVZ: gold) and the US dollar index.

In [2]:
def fred(code):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    return pd.read_csv(url, index_col=0, parse_dates=True, na_values=".")[code]

rates = pd.DataFrame({"yield10": fred("DGS10"), "real10": fred("DFII10")})
rates.tail()

,yield10,real10
observation_date,,
2026-10-14,5.19,2.84
2026-10-15,5.10,2.78
2026-10-16,5.09,2.74
2026-10-19,5.15,2.80
2026-10-20,5.12,2.77


10-year Treasury yield (DGS10) and 10-year real yield (DFII10) from FRED.

In [3]:
daily = market.join(rates).ffill()
weekly = daily.resample("W-FRI").last().loc[rv.index]

Combine, fill holidays with the last known value, and take each week's Friday value so the features line up with the volatility weeks.

In [4]:
feats = pd.DataFrame(index=rv.index)
for c in rv:
    feats[f"{c}_rv"] = rv[c]
    feats[f"{c}_rv4"] = rv[c].rolling(4).mean()
    feats[f"{c}_rv12"] = rv[c].rolling(12).mean()
for c in weekly:
    feats[c] = weekly[c]
    feats[f"{c}_chg"] = weekly[c].diff()

feats = feats.dropna()
feats.to_csv("features.csv")
feats.shape

(602, 24)

Feature table with 24 columns: the HAR terms for all four assets, plus each market series and its weekly change. Each row only uses information known at the end of that week.

In [5]:
feats.corrwith(rv.shift(-1)["oil"]).sort_values()

yield10        -0.154020
real10         -0.092489
yield10_chg    -0.041255
dollar_chg      0.024199
real10_chg      0.043129
vix_chg         0.053208
bitcoin_rv4     0.055839
dollar          0.057542
bitcoin_rv12    0.057780
gvz_chg         0.069592
bitcoin_rv      0.088905
ovx_chg         0.119348
silver_rv       0.203710
silver_rv4      0.263233
gold_rv         0.267349
silver_rv12     0.296080
gold_rv12       0.366885
oil_rv12        0.385511
gold_rv4        0.385743
gvz             0.436402
vix             0.507838
oil_rv4         0.549453
oil_rv          0.594114
ovx             0.650064
dtype: float64

Correlation of each feature with **next** week's oil volatility. OVX (0.65) beats oil's own last week (0.59): option prices carry forward-looking information.

In [6]:
import statsmodels.api as sm

def ols_forecast(y, X, start="2021"):
    X = sm.add_constant(X).shift(1)
    data = pd.concat([y.rename("y"), X], axis=1).dropna()
    preds = {}
    for t in data.loc[start:].index:
        train = data.loc[:t].iloc[:-1]
        fit = sm.OLS(train["y"], train.drop(columns="y")).fit()
        preds[t] = fit.predict(data.loc[[t]].drop(columns="y")).iloc[0]
    return pd.Series(preds)

General walk-forward OLS forecaster: the same as HAR, but the predictor table is passed in.

In [7]:
iv = {"oil": "ovx", "gold": "gvz", "silver": "gvz", "bitcoin": "vix"}

har, harx, harall = {}, {}, {}
for c in rv:
    own = [f"{c}_rv", f"{c}_rv4", f"{c}_rv12"]
    har[c] = ols_forecast(rv[c], feats[own])
    harx[c] = ols_forecast(rv[c], feats[own + [iv[c]]])
    harall[c] = ols_forecast(rv[c], feats)

har, harx, harall = pd.DataFrame(har), pd.DataFrame(harx), pd.DataFrame(harall)

Three versions per asset: plain HAR; HAR plus the asset's own implied-volatility index (HARX); HAR plus all 24 features.

In [8]:
test = rv.loc["2021":]
naive = rv.shift(1).loc["2021":]
rmse = lambda f: np.sqrt(((test - f)**2).mean())

pd.DataFrame({
    "naive": rmse(naive),
    "HAR": rmse(har),
    "HAR + implied vol": rmse(harx),
    "HAR + all features": rmse(harall),
}).round(4)

,naive,HAR,HAR + implied vol,HAR + all features
bitcoin,0.3591,0.2760,0.2764,0.2937
oil,0.1923,0.1634,0.1532,0.1988
gold,0.1010,0.0834,0.0773,0.0788
silver,0.2607,0.2023,0.2029,0.2045


The matching implied-volatility index improves oil (−6%) and gold (−7%). Adding all 24 features overfits: worse than HAR for three assets, and worse than the naive forecast for oil.

In [9]:
pd.concat({"HAR": har, "HARX": harx, "HAR_all": harall}, axis=1).to_csv("harx_forecasts.csv")

Save the forecasts.